In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime

# Geospatial
import geopandas as gpd
from shapely.geometry import Point

# --------------------------------------------------
# Pfade
# --------------------------------------------------

RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

# --------------------------------------------------
# Helper: Data Quality Report
# --------------------------------------------------

def data_quality_report(df, name):
    report = {
        "dataset": name,
        "rows": len(df),
        "missing_values": df.isna().sum().to_dict(),
        "duplicate_rows": int(df.duplicated().sum()),
        "columns": list(df.columns)
    }
    return report

# --------------------------------------------------
# Processing Pipeline
# --------------------------------------------------

reports = []

csv_files = glob.glob(os.path.join(RAW_DIR, "*.csv"))

if not csv_files:
    print("Keine CSV Dateien im raw Ordner gefunden.")

for file_path in csv_files:



    df = pd.read_csv(file_path)

    # --------------------------------------------------
    # 1. Basic Cleaning
    # --------------------------------------------------

    df = df.drop_duplicates()

    # Pflichtspalten prüfen (FIRMS Standard)
    required_cols = ["latitude", "longitude"]

    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Fehlende Spalte: {col} in {file_path}")

    # Entferne ungültige Koordinaten
    df = df.dropna(subset=["latitude", "longitude"])
    df = df[(df["latitude"].between(-90, 90)) & (df["longitude"].between(-180, 180))]

    # --------------------------------------------------
    # 2. Georeferenzierung (Point Geometry)
    # --------------------------------------------------

    geometry = [Point(xy) for xy in zip(df["longitude"], df["latitude"])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

    # --------------------------------------------------
    # 3. DBSCAN-ready Feature Engineering
    # --------------------------------------------------

    # --------------------------------------------------
    # Date & Time Standardisierung
    # --------------------------------------------------

    # acq_date -> YYYY-MM-DD
    if "acq_date" in cluster_df.columns:
        cluster_df["acq_date"] = pd.to_datetime(
        cluster_df["acq_date"],
        errors="coerce"
        ).dt.strftime("%Y-%m-%d")

    # acq_time -> HH:MM (24h)
    if "acq_time" in cluster_df.columns:
        cluster_df["acq_time"] = (
        cluster_df["acq_time"]
        .astype(str)
        .str.zfill(4)
        )

    cluster_df["acq_time"] = (
        cluster_df["acq_time"].str[:2]
        + ":"
        + cluster_df["acq_time"].str[2:]
        )

    # numerische Features für Clustering
    cluster_df = gdf.copy()

    cluster_df["lat"] = cluster_df.geometry.y
    cluster_df["lon"] = cluster_df.geometry.x

    # optional FIRMS Features
    for col in ["brightness", "frp", "confidence"]:
        if col in cluster_df.columns:
            cluster_df[col] = pd.to_numeric(cluster_df[col], errors="coerce")

    # final dataset für DBSCAN
    cluster_features = cluster_df[[
    c for c in [
        "lat",
        "lon",
        "brightness",
        "frp",
        "acq_date",
        "acq_time"]if c in cluster_df.columns]]

        # --------------------------------------------------
    # 3b. Missing Values & Quality Handling
    # --------------------------------------------------

    na_replacements = {}

    # confidence komplett entfernen
    if "confidence" in cluster_df.columns:
        cluster_df = cluster_df.drop(columns=["confidence"])

    # missing values report
    report = data_quality_report(df, os.path.basename(file_path))


    # optional feature stats
    optional_stats = {}

    for col in ["brightness", "frp", "confidence"]:
        if col in cluster_df.columns:
            optional_stats[f"{col}_min"] = cluster_df[col].min(skipna=True)
            optional_stats[f"{col}_max"] = cluster_df[col].max(skipna=True)

    report.update(optional_stats)

    reports.append(report)

    # --------------------------------------------------
    # 4. Save outputs
    # --------------------------------------------------

    base_name = os.path.basename(file_path).replace(".csv", "")

    geo_path = os.path.join(PROCESSED_DIR, f"{base_name}_geo.geojson")
    cluster_path = os.path.join(PROCESSED_DIR, f"{base_name}_cluster.csv")

    gdf.to_file(geo_path, driver="GeoJSON")
    cluster_features.to_csv(cluster_path, index=False)

    

# --------------------------------------------------
# 5. Save Data Quality Report
# --------------------------------------------------

report_df = pd.DataFrame(reports)
report_path = os.path.join(PROCESSED_DIR, f"data_quality_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")

report_df.to_csv(report_path, index=False)

print("Fertig.")


# --------------------------------------------------
# 6. Zusammenfassung ausgeben
# --------------------------------------------------

print("================ QUALITY SUMMARY ================")

for report in reports:

    print(f"Dataset: {report['dataset']}")
    print(f"Rows: {report['rows']}")
    print(f"Duplicate Rows: {report['duplicate_rows']}")

    print("Missing Values per Column:")
    for col, val in report["missing_values"].items():
        if val > 0:
            print(f"  - {col}: {val}")


    print("Feature Ranges:")

    for feature in ["brightness", "frp"]:
        min_key = f"{feature}_min"
        max_key = f"{feature}_max"

        if min_key in report and max_key in report:
            print(f"  - {feature}: min={report[min_key]} | max={report[max_key]}")

print("=================================================")

Fertig.
Report gespeichert: ..\data\processed\data_quality_report_20260521_143326.csv
================ QUALITY SUMMARY ================
Dataset: firms_VIIRS_NOAA20_NRT_south_america_20260521_092524.csv
Rows: 681
Duplicate Rows: 0
Missing Values per Column:
Feature Ranges:
  - frp: min=0.28 | max=37.05
Dataset: firms_VIIRS_NOAA21_NRT_south_america_20260521_092523.csv
Rows: 515
Duplicate Rows: 0
Missing Values per Column:
Feature Ranges:
  - frp: min=0.27 | max=40.75
Dataset: firms_VIIRS_SNPP_NRT_south_america_20260521_092525.csv
Rows: 567
Duplicate Rows: 0
Missing Values per Column:
Feature Ranges:
  - frp: min=0.22 | max=31.41
